***Differential Charging***

In this model, we are considering Secondary Electron Yield (SEY)- Backscattered Electron Yield (BEY) for the electrons that are interacting with the surface of the spacecraft.

The spacecraft isn't made of single type of material but combination of both dielectric and conductor. Assuming the spacecraft to be ***a triple junction of metal, dielectric and the surrounding ambient plasma.***

Different material will have different SEY-BEY characteristic values, that will lead to difference in potental at different surfaces of spacecraft that will lead to Differential Charging. 

If **Differential Voltage> ESD inception voltage**, then ESD occurs. (Threat to Spacecrafts!)

On conducting an experiment that includes placing the designed spacecraft model in the laboratory plasma like environment can give us Electrostatic Discharge inception Voltage (ESD) of such triple junction (need to be done)

The notebook contains an input file where it can include different input parameters for the main objective to find the Differential Voltage followed by the operational code to evaluate it and finally the output file consisting results and plots. 





In [ ]:
# ==========================================================
# SPACECRAFT CHARGING — SINGLE FACE MODEL
# 3D Laplace solver with OML surface charging
#
# Active surface : x=0 face only
#   Copper patch : Dirichlet  V = Vsc = Q/C
#   Kapton       : Neumann    eps_kap * dV/dn = sigma
# Other 5 faces  : open boundaries  dV/dn = 0
# Interior       : vacuum  nabla^2 V = 0
#
# Laplace solved exactly each timestep using
# scipy sparse direct solver (no iteration needed).
#
# *** MODIFIED: convergence-based stopping (see Section 7) ***
# *** MODIFIED: dedicated per-quantity history arrays (see Sections 6-8) ***
# ==========================================================

import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
import warnings; warnings.filterwarnings('ignore')

# ==========================================================
# 1. PLASMA PARAMETERS
# ==========================================================
ne   = 1.25e5;  ni  = 1.25e5
KTe  = 3000  * 1.602e-19   # electron thermal energy [J]
KTi  = 3000 * 1.602e-19   # ion thermal energy      [J]
me   = 9.109e-31;  mi  = 1.673e-27
e    = 1.602e-19;  eps0 = 8.854e-12
eps_kap = 4.0 * eps0       # Kapton permittivity

Je0 = ne * e * np.sqrt(KTe / (2.0 * np.pi * me))
Ji0 = ni * e * np.sqrt(KTi / (2.0 * np.pi * mi))
# Theoretical floating potential: Je(Vf)=Ji => Vf = -(KTe/e)*ln(Je0/Ji0)
Vf_theory = -(KTe / e) * np.log(Je0 / Ji0)

print(f"Je0 = {Je0:.3e} A/m2")
print(f"Ji0 = {Ji0:.3e} A/m2")
print(f"Je0/Ji0 = {Je0/Ji0:.2f}  =>  negative charging")
print(f"Theoretical floating potential = {Vf_theory:.1f} V")

# ==========================================================
# 2. GRID
# ==========================================================
Nx, Ny, Nz = 20, 20, 20      #we can incresease this for higher resolution
Lx, Ly, Lz = 1.0, 1.0, 1.0  # domain size [m]

x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
z = np.linspace(0, Lz, Nz)
dx = x[1]-x[0];  dy = y[1]-y[0];  dz = z[1]-z[0]
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
N = Nx * Ny * Nz

def idx(i, j, k):
    """Flat index into the solution vector."""
    return i*Ny*Nz + j*Nz + k

# ==========================================================
# 3. MASKS
# ==========================================================
# Copper patch centred on x=0 face
copper_mask = np.zeros((Nx, Ny, Nz), dtype=bool)
yc = Ny//2;  zc = Nz//2;  hy = 3;  hz = 3
copper_mask[0, yc-hy:yc+hy, zc-hz:zc+hz] = True

# Kapton = rest of x=0 face
xface = np.zeros((Nx, Ny, Nz), dtype=bool);  xface[0, :, :] = True
kapton_mask = xface & (~copper_mask)

print(f"\nGrid  : {Nx}x{Ny}x{Nz}   dx={dx:.4f} m")
print(f"Copper: {copper_mask.sum()} nodes")
print(f"Kapton: {kapton_mask.sum()} nodes")

# ==========================================================
# 4. BUILD SPARSE MATRIX
# ==========================================================
# Boundary condition derivation for each face:
#
# x=0 Kapton (outward normal n = -x):
#   eps_kap * dV/dn = sigma
#   dV/dn = -dV/dx = -(V[1]-V[0])/dx
#   => V[0] - V[1] = sigma*dx/eps_kap     [row: A*1 - A*(-1) = rhs]
#
# x=Lx, y=0, y=Ly, z=0, z=Lz (open, dV/dn=0):
#   boundary node = its nearest interior neighbour
#   V[boundary] - V[interior_neighbour] = 0
#
# Interior (6-point Laplacian):
#   -6V[i,j,k] + V[i+1]+V[i-1]+V[j+1]+V[j-1]+V[k+1]+V[k-1] = 0

A = lil_matrix((N, N))
for i in range(Nx):
    for j in range(Ny):
        for k in range(Nz):
            n = idx(i, j, k)
            if copper_mask[i, j, k]:          # Dirichlet
                A[n, n] = 1.
            elif kapton_mask[i, j, k]:        # Neumann eps_kap*dV/dn=sigma
                A[n, idx(0, j, k)] = 1.
                A[n, idx(1, j, k)] = -1.
            elif i == Nx-1:                   # open: x=Lx
                A[n, idx(Nx-1, j, k)] = 1.;  A[n, idx(Nx-2, j, k)] = -1.
            elif j == 0:                      # open: y=0
                A[n, idx(i, 0, k)] = 1.;     A[n, idx(i, 1, k)] = -1.
            elif j == Ny-1:                   # open: y=Ly
                A[n, idx(i, Ny-1, k)] = 1.;  A[n, idx(i, Ny-2, k)] = -1.
            elif k == 0:                      # open: z=0
                A[n, idx(i, j, 0)] = 1.;     A[n, idx(i, j, 1)] = -1.
            elif k == Nz-1:                   # open: z=Lz
                A[n, idx(i, j, Nz-1)] = 1.;  A[n, idx(i, j, Nz-2)] = -1.
            else:                             # interior Laplacian
                A[n, n] = -6.
                A[n, idx(i+1,j,k)]=1.; A[n, idx(i-1,j,k)]=1.
                A[n, idx(i,j+1,k)]=1.; A[n, idx(i,j-1,k)]=1.
                A[n, idx(i,j,k+1)]=1.; A[n, idx(i,j,k-1)]=1.
A = A.tocsr()
print("Sparse matrix built (factorised once per solve).")

# ==========================================================
# 5. SIMULATION PARAMETERS
# ==========================================================
C_sc  = 312.8e-12   # spacecraft capacitance [F]
j_tol = 1e-9     # steady-state current density tolerance [A/m2]
V_tol = 1e-2        # steady-state potential-change tolerance [V]
dt    = 0.00001       # time step [s]
t_end = 2.0       # maximum simulation time [s]  (upper bound, not a target)

# ==========================================================
# 6. INITIAL CONDITIONS & HISTORY
# ==========================================================
sigma    = np.zeros((Nx, Ny, Nz))
Q_copper = 0.0
V        = np.zeros((Nx, Ny, Nz))

time_h=[0.]; Vcu_h=[0.]; Vkap_h=[0.]; sig_h=[0.]

# --- Dedicated convergence-history arrays (one entry per time step, same length as time_h) ---
mean_Jkap_h      = [0.]   # mean net current density on Kapton   [A/m2]
max_Jkap_h       = [0.]   # max  net current density on Kapton   [A/m2]
max_Jcop_h       = [0.]   # max  net current density on Copper   [A/m2]
max_dV_kapton_h  = [0.]   # max |V(n+1)-V(n)| restricted to Kapton nodes [V]
max_dV_copper_h  = [0.]   # max |V(n+1)-V(n)| restricted to Copper nodes [V]

print(f"\nt=0.0s | V_copper=0.0V | V_kapton=0.0V  [initial]")

# ==========================================================
# 7. TIME LOOP  (convergence-based stopping)
# ==========================================================
t = dt
step_count = 0          # counts how many time steps were actually taken
converged  = False       # flag so we know, after the loop, why it stopped

while t <= t_end + 0.5*dt:
    step_count += 1
    print(t)
    # ---- remember previous-step potential field for the ΔV check ----
    V_prev = V.copy()

    # ---- OML on Kapton ----
    V_kap   = V[kapton_mask]
    je_kap  = Je0 * np.exp(np.clip(V_kap*e/KTe, -200, 50))
    ji_kap  = Ji0 * np.maximum(0., 1. - V_kap*e/KTi)
    jnet_kap = ji_kap - je_kap
    sigma[kapton_mask] += jnet_kap * dt         # dσ/dt = Jnet

    # ---- OML on Copper ----
    V_cop   = V[copper_mask]
    je_cop  = Je0 * np.exp(np.clip(V_cop*e/KTe, -200, 50))
    ji_cop  = Ji0 * np.maximum(0., 1. - V_cop*e/KTi)
    jnet_cop = ji_cop - je_cop                  # kept as an array (reused below)
    Q_copper += np.sum(jnet_cop) * dy*dz * dt
    V_sc = Q_copper / C_sc

    # ---- Build RHS and solve ----
    b = np.zeros(N)
    for i in range(Nx):
        for j in range(Ny):
            for k in range(Nz):
                n = idx(i, j, k)
                if copper_mask[i, j, k]:
                    b[n] = V_sc
                elif kapton_mask[i, j, k]:
                    b[n] = sigma[i, j, k] * dx / eps_kap
                # all other nodes: b[n] = 0
    V = spsolve(A, b).reshape(Nx, Ny, Nz)

    # ---- Compute all convergence measures ONCE per step ----
    max_Jnet_kap    = np.max(np.abs(jnet_kap))
    max_Jnet_cop    = np.max(np.abs(jnet_cop))
    dV_full         = V - V_prev                              # computed once, sliced below (no recomputation)
    max_dV_kapton   = np.max(np.abs(dV_full[kapton_mask]))
    max_dV_copper   = np.max(np.abs(dV_full[copper_mask]))
    max_dV          = max(max_dV_kapton, max_dV_copper)       # combined measure used for the convergence test

    # ---- Record: exactly one append per array, per time step ----
    Vcu  = np.mean(V[copper_mask])
    Vkap = np.mean(V[kapton_mask])
    time_h.append(t); Vcu_h.append(Vcu); Vkap_h.append(Vkap)
    sig_h.append(np.mean(sigma[kapton_mask]))
    mean_Jkap_h.append(np.mean(jnet_kap))
    max_Jkap_h.append(max_Jnet_kap)
    max_Jcop_h.append(max_Jnet_cop)
    max_dV_kapton_h.append(max_dV_kapton)
    max_dV_copper_h.append(max_dV_copper)

    if abs(t % 0.01) < dt:
        print(f"t={t:.3f}s | V_cu={Vcu:+.1f}V | V_kap={Vkap:+.1f}V | "
          f"max|Jnet_kap|={max_Jnet_kap:.3e} | max|Jnet_cop|={max_Jnet_cop:.3e} | "
          f"max|dV|={max_dV:.3e}")

    # ---- Joint convergence check ----
    if (max_Jnet_kap < j_tol) and (max_Jnet_cop < j_tol) and (max_dV < V_tol):
        converged = True
        print(f"\nConvergence achieved at t = {t:.2f} s after {step_count} time steps.")
        print(f"  max|Jnet_Kapton| = {max_Jnet_kap:.3e} A/m2  (tol = {j_tol:.1e})")
        print(f"  max|Jnet_Copper| = {max_Jnet_cop:.3e} A/m2  (tol = {j_tol:.1e})")
        print(f"  max|dV_Kapton|   = {max_dV_kapton:.3e} V     (tol = {V_tol:.1e})")
        print(f"  max|dV_Copper|   = {max_dV_copper:.3e} V     (tol = {V_tol:.1e})")
        break

    t += dt

# ---- Explicit message if t_end was reached without convergence ----
if not converged:
    print("\nSimulation reached t_end before satisfying the convergence criteria. "
          "Increase t_end or modify the time step if a steady-state solution is required.")

print(f"\nTheory Vf      = {Vf_theory:.1f} V")
print(f"Simulated Vkap = {Vkap_h[-1]:.1f} V")
print(f"Simulated Vcu  = {Vcu_h[-1]:.1f} V")

# ==========================================================
# 8. POST-PROCESSING
# ==========================================================
time_h=np.array(time_h); Vcu_h=np.array(Vcu_h)
Vkap_h=np.array(Vkap_h); sig_h=np.array(sig_h)
mean_Jkap_h     = np.array(mean_Jkap_h)
max_Jkap_h      = np.array(max_Jkap_h)
max_Jcop_h      = np.array(max_Jcop_h)
max_dV_kapton_h = np.array(max_dV_kapton_h)
max_dV_copper_h = np.array(max_dV_copper_h)
En = (V[1,:,:] - V[0,:,:]) / dx   # normal E at x=0 face

ylo=(yc-hy)*dy; yhi=(yc+hy)*dy; zlo=(zc-hz)*dz; zhi=(zc+hz)*dz
def add_copper(ax):
    ax.add_patch(plt.Rectangle((ylo,zlo),yhi-ylo,zhi-zlo,
                 edgecolor='gold',facecolor='none',lw=2,label='Copper patch'))

# Figure 1: Time histories
fig,axes=plt.subplots(2,2,figsize=(12,7))
fig.suptitle('Single Face Spacecraft Charging — Time Histories',fontsize=13,fontweight='bold')
ax=axes[0,0]
ax.plot(time_h,Vcu_h,'b-',lw=2,label='Copper')
ax.plot(time_h,Vkap_h,'r--',lw=2,label='Kapton mean')
ax.axhline(Vf_theory,color='gray',ls=':',lw=1.5,label=f'Theory Vf={Vf_theory:.0f}V')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Potential [V]')
ax.set_title('Surface potentials vs time'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax=axes[0,1]; ax.plot(time_h[1:],mean_Jkap_h[1:],'g-',lw=2); ax.axhline(0,color='k',lw=0.8,ls='--')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Mean Jnet [A/m2]'); ax.set_title('Net current on Kapton'); ax.grid(alpha=0.3)
ax=axes[1,0]; ax.plot(time_h[1:],sig_h[1:],'m-',lw=2)
ax.set_xlabel('Time [s]'); ax.set_ylabel('Mean sigma [C/m2]'); ax.set_title('Kapton surface charge density'); ax.grid(alpha=0.3)
ax=axes[1,1]; ax.plot(time_h,Vcu_h-Vkap_h,'k-',lw=2)
ax.set_xlabel('Time [s]'); ax.set_ylabel('Delta V [V]'); ax.set_title('Differential charging: V_copper - V_kapton'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('sf_fig1.png',dpi=150,bbox_inches='tight')
plt.show()
# Figure 2: x=0 face surface maps
fig,axes=plt.subplots(1,3,figsize=(15,5))
fig.suptitle('x=0 Face — Final State',fontsize=13,fontweight='bold')
im=axes[0].contourf(Y[0,:,:],Z[0,:,:],V[0,:,:],levels=40,cmap='RdBu_r')
plt.colorbar(im,ax=axes[0],label='V [V]'); add_copper(axes[0]); axes[0].legend(fontsize=8)
axes[0].set_xlabel('y [m]'); axes[0].set_ylabel('z [m]'); axes[0].set_title('Surface potential [V]')
sp=np.where(kapton_mask[0,:,:],sigma[0,:,:],np.nan)
im=axes[1].contourf(Y[0,:,:],Z[0,:,:],sp,levels=40,cmap='plasma')
plt.colorbar(im,ax=axes[1],label='sigma [C/m2]'); add_copper(axes[1])
axes[1].set_xlabel('y [m]'); axes[1].set_ylabel('z [m]'); axes[1].set_title('Surface charge density')
im=axes[2].contourf(Y[0,:,:],Z[0,:,:],En,levels=40,cmap='coolwarm')
plt.colorbar(im,ax=axes[2],label='En [V/m]'); add_copper(axes[2])
axes[2].set_xlabel('y [m]'); axes[2].set_ylabel('z [m]'); axes[2].set_title('Normal E-field [V/m]')
plt.tight_layout(); plt.savefig('sf_fig2.png',dpi=150,bbox_inches='tight')
plt.show()
# Figure 3: vacuum field interior slices
fig,axes=plt.subplots(1,2,figsize=(12,5))
fig.suptitle('Vacuum Potential Field — Interior Slices',fontsize=13,fontweight='bold')
mid_y=Ny//2; mid_z=Nz//2; vmin=V.min(); vmax=V.max()
im=axes[0].contourf(X[:,mid_y,:],Z[:,mid_y,:],V[:,mid_y,:],levels=40,cmap='RdBu_r',vmin=vmin,vmax=vmax)
plt.colorbar(im,ax=axes[0],label='V [V]'); axes[0].set_xlabel('x [m]'); axes[0].set_ylabel('z [m]')
axes[0].set_title(f'y={y[mid_y]:.2f}m slice'); axes[0].axvline(0,color='k',lw=1.5,ls='--',label='Active face x=0'); axes[0].legend(fontsize=8)
im=axes[1].contourf(X[:,:,mid_z],Y[:,:,mid_z],V[:,:,mid_z],levels=40,cmap='RdBu_r',vmin=vmin,vmax=vmax)
plt.colorbar(im,ax=axes[1],label='V [V]'); axes[1].set_xlabel('x [m]'); axes[1].set_ylabel('y [m]'); axes[1].set_title(f'z={z[mid_z]:.2f}m slice')
plt.tight_layout(); plt.savefig('sf_fig3.png',dpi=150,bbox_inches='tight')
plt.show()
# Figure 4: potential decay into vacuum
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(x,V[:,yc,zc],'b-',lw=2,label='Centre of copper patch')
ax.plot(x,V[:,yc-hy-1,zc],'r--',lw=2,label='Kapton near copper edge')
ax.plot(x,V[:,1,1],'g-.',lw=2,label='Kapton far corner')
ax.axhline(Vf_theory,color='gray',ls=':',lw=1.2,label=f'Theory Vf={Vf_theory:.0f}V')
ax.set_xlabel('x [m]  (x=0: active face  |  x=1: open BC)')
ax.set_ylabel('V [V]'); ax.set_title('Potential decay into vacuum'); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('sf_fig4.png',dpi=150,bbox_inches='tight')
plt.show()

Je0 = 1.835e-07 A/m2
Ji0 = 4.282e-09 A/m2
Je0/Ji0 = 42.86  =>  negative charging
Theoretical floating potential = -11273.5 V

Grid  : 10x10x10   dx=0.1111 m
Copper: 36 nodes
Kapton: 64 nodes
Sparse matrix built (factorised once per solve).

t=0.0s | V_copper=0.0V | V_kapton=0.0V  [initial]
1e-05
2e-05
3.0000000000000004e-05
4e-05
5e-05
6e-05
7.000000000000001e-05
8e-05
9e-05
0.0001
0.00011
0.00012
0.00013000000000000002
0.00014000000000000001
0.00015000000000000001
0.00016
0.00017
0.00018
0.00019
0.0002
0.00021
0.00022
0.00023
0.00024
0.00025
0.00026000000000000003
0.00027000000000000006
0.0002800000000000001
0.0002900000000000001
0.00030000000000000014
0.00031000000000000016
0.0003200000000000002
0.0003300000000000002
0.00034000000000000024
0.00035000000000000027
0.0003600000000000003
0.0003700000000000003
0.00038000000000000035
0.00039000000000000037
0.0004000000000000004
0.0004100000000000004
0.00042000000000000045
0.0004300000000000005
0.0004400000000000005
0.00045000000000000053
0

KeyboardInterrupt: 